# Phase 2 — Path C: probability and distributions
Reproducible continuation of Phase 1; all probabilities refer to the supplied, imputed, matched participant file, not a target population. Calculations below are explicit. Run from the repository root or notebooks directory.

In [1]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from IPython.display import display
root = Path.cwd()
if not (root / 'data').exists(): root = root.parent
u = pd.read_csv(root/'data/udslong_imputed.csv')
m = pd.read_csv(root/'data/mriscanclean.csv')
assert u.NACCID.notna().all() and m.NACCID.notna().all()
assert u.NACCID.is_unique and m.NACCID.is_unique
d0 = u.merge(m, on='NACCID', validate='one_to_one', suffixes=('_uds','_mri'))
d = d0.dropna(subset=['CDRGLOB','LH_ENTORHINAL_AVGTH']).copy()
d = d[d.CDRGLOB.isin([0, .5, 1, 2])].copy()
n = len(d)
assert n > 0
print('Joined:',len(d0),'Eligible:',n)


Joined: 1237 Eligible: 1237


## Shared Task 1 — Sample space and random variables
Select one eligible matched participant uniformly: Ω is the 1,237 eligible NACCIDs and each observation has mass 1/n. A means that participant has recorded CDRGLOB > 0; B means their left entorhinal thickness is strictly below the eligible-sample median. A and B are binary indicators (0 or 1), appropriate for joint and conditional frequencies; B is a sample-defined imaging grouping, not a clinical cutoff. X is that same participant’s recorded CDRSUM, a quantitative clinical score used to describe its empirical distribution and covariance with thickness. Left thickness is a quantitative, continuous-valued MRI measurement; NACCAPOE is a recorded categorical code, not a validated genotype. The finite empirical distribution is not a population model. Eligibility restricts CDRGLOB to observed values in this file, not codebook validation. Median ties are in not-B.

In [2]:
threshold = d.LH_ENTORHINAL_AVGTH.median()
A = d.CDRGLOB > 0
B = d.LH_ENTORHINAL_AVGTH < threshold
X = d.CDRSUM
print('n =', n, 'threshold =', threshold, 'CDRGLOB values:', sorted(d.CDRGLOB.unique()))

n = 1237 threshold = 3.021 CDRGLOB values: [0.0, 0.5, 1.0, 2.0]


## Shared Task 2 — Marginal probabilities
Frequency divided by the number of eligible participants.

In [3]:
print('P(A) =', A.sum(), '/', n, '=', A.mean()); print('P(B) =', B.sum(), '/', n, '=', B.mean())

P(A) = 437 / 1237 = 0.3532740501212611
P(B) = 615 / 1237 = 0.4971705739692805


## Shared Task 3 — Joint and conditional probabilities
P(A∩B) uses n; conditional probabilities use the conditioning-event count. Both conditioning events are nonempty.

In [4]:
joint = int((A & B).sum()); a = int(A.sum()); b = int(B.sum())
assert a and b and n-b
print(pd.crosstab(B.rename('B'), A.rename('A'), margins=True))
print('P(A∩B) =', joint, '/', n, '=', joint/n)
print('P(A|B) =', joint, '/', b, '=', joint/b)
print('P(A|not B) =', a-joint, '/', n-b, '=', (a-joint)/(n-b))

A      False  True   All
B                       
False    469   153   622
True     331   284   615
All      800   437  1237
P(A∩B) = 284 / 1237 = 0.22958771220695232
P(A|B) = 284 / 615 = 0.46178861788617886
P(A|not B) = 153 / 622 = 0.2459807073954984


## Shared Task 4 — Empirical independence
Compare the joint probability with the product of the marginals. This is an exact statement about the empirical distribution, not a hypothesis test or a causal claim.

In [5]:
print('joint =', joint/n, '; product =', (a/n)*(b/n), '; difference =', joint/n-(a/n)*(b/n))

joint = 0.22958771220695232 ; product = 0.17563746226723975 ; difference = 0.05395024993971256


## Shared Task 5 — Discrete distribution
For K, the number of A events in 10 hypothetical independent draws **with replacement** from the empirical distribution, K ~ Binomial(10,p), p = P(A). This is not an observed grouping of 10 people. For distinct participants sampled without replacement the finite-population distribution is hypergeometric. Independence and common event probability are model assumptions, not guaranteed by unique IDs.

In [6]:
p = a/n
pmf = pd.Series({k: math.comb(10,k)*p**k*(1-p)**(10-k) for k in range(11)}, name='probability')
print('E(K)=',10*p,'Var(K)=',10*p*(1-p),'P(K≥4)=',pmf.loc[4:].sum())
display(pmf.to_frame()); pmf.plot.bar(title='Hypothetical binomial count', xlabel='K', ylabel='Probability'); plt.show()

E(K)= 3.5327405012126114 Var(K)= 2.2847149563218183 P(K≥4)= 0.4950557914462811


,probability
0,0.012800
1,0.069919
2,0.171869
3,0.250356
4,0.239325
5,0.156878
6,0.071412
7,0.022291
8,0.004566
9,0.000554


## Shared Task 6 — Distribution shape
Histogram and normal Q–Q plot describe observed left thickness. Visual departures, especially tails, limit a normal approximation. CDRSUM has substantial mass at zero and is not well represented by a normal distribution.

In [7]:
fig, ax = plt.subplots(1,3,figsize=(12,3))
ax[0].hist(d.CDRSUM.dropna(),bins=25); ax[0].set_title('CDRSUM')
x = np.sort(d.LH_ENTORHINAL_AVGTH.dropna().to_numpy())
ax[1].hist(x,bins=25); ax[1].set_title('Left thickness')
z = norm.ppf((np.arange(len(x))+.5)/len(x))
ax[2].scatter(z,x,s=5); ax[2].plot(z,x.mean()+x.std()*z,color='red'); ax[2].set_title('Normal Q–Q')
plt.tight_layout(); plt.show()

## Shared Task 7 — Empirical expectation and variance
For observed values xᵢ, E(X)=Σxᵢ/n and Var(X)=Σ(xᵢ−E(X))²/n. Drop explicit missing values per variable; no undocumented code is recoded. These are empirical-distribution moments, not standard errors.

In [8]:
for name in ['CDRSUM','LH_ENTORHINAL_AVGTH','RH_ENTORHINAL_AVGTH']:
    x = d[name].dropna().to_numpy(); mu = sum(x)/len(x)
    print(name, 'n =',len(x),'E =',mu,'Var =',sum((x-mu)**2)/len(x))

CDRSUM n = 1237 E = 0.8221503637833468 Var = 2.9663484883042477
LH_ENTORHINAL_AVGTH n = 1237 E = 2.9697987065480977 Var = 0.14745001687787426
RH_ENTORHINAL_AVGTH n = 1237 E = 3.0577267582861802 Var = 0.16088274344729242


## Shared Task 8 — Covariance
On complete pairs, Cov(X,Y)=mean((X−mean X)(Y−mean Y)); denominator is the number of pairs. Sign describes co-variation, units are scale-dependent. No correlation or regression is computed.

In [9]:
for side in ['LH','RH']:
    pair = d[[side+'_ENTORHINAL_AVGTH','CDRSUM']].dropna()
    x,y = pair.iloc[:,0],pair.iloc[:,1]
    print(side, 'paired n =',len(pair), 'covariance =',((x-x.mean())*(y-y.mean())).mean())

LH paired n = 1237 covariance = -0.17747678622426677
RH paired n = 1237 covariance = -0.18015005107279


## Shared Task 9 — Bayes identity
With A as the event of interest and B as the observed information, use P(A|B)=P(B|A)P(A)/P(B). This rearranges the same table; it is not an external prior or validated clinical prediction.

In [10]:
print('P(B|A) =',joint,'/',a,'=',joint/a); print('Bayes =',(joint/a)*(a/n)/(b/n),'direct =',joint/b); assert np.isclose((joint/a)*(a/n)/(b/n),joint/b)

P(B|A) = 284 / 437 = 0.6498855835240275
Bayes = 0.4617886178861789 direct = 0.46178861788617886


## C1 — Cognitive status in two groupings
First group participants by MRI-derived B versus not-B (median ties in not-B); then independently group by recorded NACCAPOE code. For each grouping report A counts, denominators and proportions. No NACCAPOE code, including 9, is assigned genotype meaning; small groups are unstable. These are descriptive within the same eligible matched sample, not causal or inferential comparisons.

In [11]:
display(d.assign(A=A, MRI_group=np.where(B, 'below median', 'at or above median')).groupby('MRI_group').A.agg(['sum','count','mean']))

,sum,count,mean
MRI_group,,,
at or above median,153,622,0.245981
below median,284,615,0.461789


In [12]:
groups = d.assign(A=A).groupby('NACCAPOE',dropna=False).A.agg(['sum','count','mean']); display(groups)

,sum,count,mean
NACCAPOE,,,
1.0,129,493,0.261663
2.0,115,290,0.396552
3.0,23,83,0.277108
4.0,44,73,0.602740
5.0,4,17,0.235294
6.0,0,3,0.000000
9.0,122,278,0.438849


## C2 — Cognitive and imaging distributions
Show quantiles and Tukey flags; flags are not automatic exclusions. MOCATOTS is examined as recorded, but its extreme and fractional values cannot be resolved without a codebook and imputation metadata; it is not substituted for a validated score.

In [13]:
for col in ['CDRGLOB','CDRSUM','MOCATOTS','LH_ENTORHINAL_AVGTH','RH_ENTORHINAL_AVGTH']:
    s=d[col].dropna(); q1,q3=s.quantile([.25,.75]); flags=((s<q1-1.5*(q3-q1)) | (s>q3+1.5*(q3-q1))).sum()
    print(col, 'n',len(s),'median',s.median(),'range', (s.min(),s.max()), 'Tukey flags',flags)
print('MOCATOTS > 30:', (d.MOCATOTS>30).sum(), '; fractional:', ((d.MOCATOTS%1!=0)&d.MOCATOTS.notna()).sum(), '; 88:', (d.MOCATOTS==88).sum())

CDRGLOB n 1237 median 0.0 range (0.0, 2.0) Tukey flags 9
CDRSUM n 1237 median 0.0 range (0.0, 15.0) Tukey flags 115
MOCATOTS n 1237 median 25.4 range (0.0, 88.0) Tukey flags 84
LH_ENTORHINAL_AVGTH n 1237 median 3.021 range (0.878, 3.979) Tukey flags 28
RH_ENTORHINAL_AVGTH n 1237 median 3.097 range (0.729, 3.967) Tukey flags 30
MOCATOTS > 30: 9 ; fractional: 249 ; 88: 4


## C3 — Empirical clinical/imaging events
The median threshold is internal to this sample, not a clinical cutoff. Comparison is descriptive and cannot establish temporal ordering.

In [14]:
print('Below-median group:',joint,'/',b,'=',joint/b); print('Other group:',a-joint,'/',n-b,'=',(a-joint)/(n-b))

Below-median group: 284 / 615 = 0.46178861788617886
Other group: 153 / 622 = 0.2459807073954984


## C4 — Bilateral imaging and CDRSUM
Scatterplots and covariance only: negative covariance indicates larger thickness tends to co-occur with lower recorded CDRSUM in these paired data, not causation.

In [15]:
fig,axes=plt.subplots(1,2,figsize=(10,4))
for ax,side in zip(axes,['LH','RH']):
    pair=d[[side+'_ENTORHINAL_AVGTH','CDRSUM']].dropna(); x=pair.iloc[:,0]; y=pair.iloc[:,1]
    ax.scatter(x,y,s=8,alpha=.3); ax.set(xlabel=side+' thickness',ylabel='CDRSUM',title='n=%d, covariance=%.4f'%(len(pair),((x-x.mean())*(y-y.mean())).mean()))
plt.tight_layout(); plt.show()

## C5 — Participant IDs and repeated observations
Inspect both inputs before joining. One supplied row per ID is the analysis unit; this does not prove participant independence, exclude historical visits, or establish temporal alignment of MRI and UDS. No chronology-based index visit can be selected from the supplied documentation.

In [16]:
for name, frame in [('UDS imputed',u),('MRI',m)]:
    print(name, 'rows',len(frame),'missing IDs',frame.NACCID.isna().sum(),'unique IDs',frame.NACCID.nunique(),'duplicate rows',frame.NACCID.duplicated().sum())
print('UDS-only',len(set(u.NACCID)-set(m.NACCID)),'MRI-only',len(set(m.NACCID)-set(u.NACCID)),'matched',len(d0))
assert d0.NACCID.is_unique

UDS imputed rows 1237 missing IDs 0 unique IDs 1237 duplicate rows 0
MRI rows 1237 missing IDs 0 unique IDs 1237 duplicate rows 0
UDS-only 0 MRI-only 0 matched 1237


## Task coverage
| Task | Status | Notebook location | Report location |
|---|---|---|---|
| Shared 1 | Addressed | Shared Task 1 | Shared Task 1 |
| Shared 2 | Addressed | Shared Task 2 | Shared Task 2 |
| Shared 3 | Addressed | Shared Task 3 | Shared Task 3 |
| Shared 4 | Addressed | Shared Task 4 | Shared Task 4 |
| Shared 5 | Addressed | Shared Task 5 | Shared Task 5 |
| Shared 6 | Addressed | Shared Task 6 | Shared Task 6 |
| Shared 7 | Addressed | Shared Task 7 | Shared Task 7 |
| Shared 8 | Addressed | Shared Task 8 | Shared Task 8 |
| Shared 9 | Addressed | Shared Task 9 | Shared Task 9 |
| C1 | Addressed | C1 | Path C Task C1 |
| C2 | Addressed | C2 | Path C Task C2 |
| C3 | Addressed | C3 | Path C Task C3 |
| C4 | Addressed | C4 | Path C Task C4 |
| C5 | Addressed | C5 | Path C Task C5 |

## Scope and limitations
UDS was already imputed; original missingness is unknown. No authoritative codebook or reliable visit/scan timing was supplied. MRI-selected participants need not represent the wider population. No Phase 3–5 testing, inference, correlation, regression, or sensitivity analysis is performed. Results are noncausal.